# 평가자 정합성 보고서 템플릿

저장된 Run의 조건·지표·질문·답변·근거·판정을 기계적으로 표시합니다.
해석·결론을 새로 작성하거나 생성하지 않습니다. 기본 입력은 실제 MLflow Run입니다.
템플릿을 `../reports/`로 복사해 입력 Run과 표시 범위를 설정하세요.
HTML에는 `report_view`가 포함되며 hover·표 검색·정렬은 export 후에도 동작합니다.

In [ ]:
import os

import holoviews as hv
import mlflow
import pandas as pd
import panel as pn
from IPython.display import display

from elice_reporting import (
    analysis_frames,
    demo,
    export_html,
    find_root,
    from_files,
    from_mlflow,
)
from elice_reporting.analysis_sections import evaluation_conditions, item_browser, metric_guide
from elice_reporting.analysis_visuals import agreement_browser, analysis_subjects

pn.extension()
hv.extension("bokeh")
ROOT = find_root()

In [ ]:
# 보고서마다 수정하는 입력입니다. 인증정보는 코드에 쓰지 않습니다.
SOURCE_MODE = "mlflow"  # demo | mlflow | local
RUN_ID = "a1ed1c497f8b4b9caf2134d043516ba9"  # analysis 또는 comparison run ID
REPORT_TITLE = "Gold 4문항 · 모델별 3회 반복 평가"
OUTPUT_HTML = ROOT / "artifacts/reports/conditions-gold-four-3round-analysis.html"
LOCAL_REPORT = ROOT / "artifacts/evaluation/agreement.json"
LOCAL_INPUTS = ROOT / "artifacts/evaluation/analysis-inputs.json"

CASE_IDS = []  # 비워 두면 저장 순서대로 전체 문항을 표시합니다.

In [ ]:
if SOURCE_MODE == "demo":
    data = demo("analysis")
elif SOURCE_MODE == "mlflow":
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    client = mlflow.MlflowClient()
    if not RUN_ID:
        experiment = client.get_experiment_by_name(
            os.environ.get("QA_MLFLOW_EXPERIMENT_NAME", "elice-techblog-rag-evaluation")
        )
        if experiment is None:
            raise ValueError("실험을 찾을 수 없습니다. 환경 설정을 확인하세요.")
        runs = client.search_runs(
            [experiment.experiment_id],
            filter_string="tags.`evaluation.kind` = 'analysis' AND attributes.status = 'FINISHED'",
            max_results=20,
        )
        display(
            pd.DataFrame(
                [{"run_id": r.info.run_id, "name": r.data.tags.get("mlflow.runName")} for r in runs]
            )
        )
        raise ValueError("위 목록에서 RUN_ID를 설정하고 다시 실행하세요.")
    data = from_mlflow("analysis", RUN_ID, ROOT / "artifacts/reports/cache")
else:
    if SOURCE_MODE != "local":
        raise ValueError("SOURCE_MODE는 demo, mlflow, local 중 하나여야 합니다.")
    data = from_files("analysis", LOCAL_REPORT, LOCAL_INPUTS)

# SDK로 metric/param을 추가 조회할 수도 있습니다(읽기 전용).
# run = client.get_run(RUN_ID)
# run.data.metrics

In [ ]:
frames = analysis_frames(data)
IS_DEMO = data.source == "demo" or data.report.get("demo", False)
IS_SYNTHETIC = bool(data.report.get("synthetic_grading_run_ids"))
banner = pn.pane.Markdown(
    "## 개발용 모의 검수 — 임의 점수이며 실제 사람 평가가 아닙니다."
    if IS_SYNTHETIC
    else "## 데모 데이터 — 실제 실험 결과가 아닙니다."
    if IS_DEMO
    else "",
    styles={"color": "#a44a08"},
)
subjects = analysis_subjects(data)

## 1. 평가 조건

저장된 데이터셋·QA 프로필과 평가 주체·회차·모델 정보를 표시합니다.

In [ ]:
conditions_view = evaluation_conditions(data, subjects)
definitions_view = metric_guide(data)

## 2. 항목별 선형 가중 Cohen κ

κ는 대칭이므로 대각선과 하삼각을 생략하고 우상단만 표시합니다. Gold는 드롭다운에서 전체 항목(2×2) 또는 개별 항목을 선택합니다.

In [ ]:
primary_view = agreement_browser(data, subjects)

## 3. 문항별 평가

드롭다운으로 문항을 선택합니다. 질문·답변과 평가 주체별 N행 × 고정 4열의 판정을 표시합니다.

In [ ]:
available_ids = frames["cases"]["record_id"].tolist()
case_ids = list(dict.fromkeys(CASE_IDS)) if CASE_IDS else available_ids
if not set(case_ids) <= set(available_ids):
    raise ValueError("CASE_IDS에 저장된 답변에 없는 ID가 있습니다.")
examples_view = item_browser(data, subjects, case_ids)

## 4. 보고서 조합

평가 조건의 원본 Run 링크로 상세 기록을 확인합니다.

In [ ]:
report_view = pn.Column(
    pn.pane.Markdown("# " + REPORT_TITLE),
    banner,
    pn.pane.Markdown("## 1. 평가 조건"),
    conditions_view,
    pn.pane.Markdown("## 2. 지표 정의와 판정 기준"),
    definitions_view,
    pn.pane.Markdown("## 3. 항목별 선형 가중 Cohen κ"),
    primary_view,
    pn.pane.Markdown("## 4. 문항별 평가"),
    examples_view,
    sizing_mode="stretch_width",
    max_width=1100,
    styles={"margin": "0 auto"},
)
report_view

## 독립 HTML export

`report_view`에 포함한 조건·차트·표·문항별 기록을 로컬 HTML로 저장합니다.
업로드는 수행하지 않으며 MLflow 첨부 예제는 주석 상태입니다.

In [ ]:
html_path = export_html(report_view, OUTPUT_HTML, title=REPORT_TITLE, overwrite=True)
html_path

# MLflow에 첨부하려면 결과와 대상 run을 확인한 뒤 직접 실행하세요.
# client.log_artifact(RUN_ID, str(html_path), artifact_path="reports")